In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [9]:
# Split qa_pairs.csv into smaller chunks to stay within the 90,000 enqueued tokens limit
import pandas as pd

# Load the QA pairs
qa_pairs_file = "/workspace/data/qa/llm_batch/qa_pairs.csv"
qa_pairs = pd.read_csv(qa_pairs_file)

# Define the maximum number of rows per chunk
max_rows_per_chunk = 250  # Adjust this value based on your token estimation

# Split the dataframe into chunks
chunk_dir = "/workspace/data/qa/llm_batch/chunks"
Path(chunk_dir).mkdir(parents=True, exist_ok=True)

for i, chunk in enumerate(range(0, len(qa_pairs), max_rows_per_chunk)):
    chunk_file = f"{chunk_dir}/qa_pairs_chunk_{i + 1}.csv"
    qa_pairs.iloc[chunk:chunk + max_rows_per_chunk].to_csv(chunk_file, index=False)
    print(f"Chunk saved to: {chunk_file}")

Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_1.csv
Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_2.csv
Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_3.csv
Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_4.csv
Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_5.csv
Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_6.csv
Chunk saved to: /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_7.csv


In [40]:
# Prepare LLM batch inputs
!python /workspace/src/evaluation/benchmark_runner_batch.py prepare \
  /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_7.csv \
  /workspace/data/results/llm_batch/gpt-4o/chunk_7/requests.jsonl \
  /workspace/data/results/llm_batch/gpt-4o/chunk_7/mapping.csv \
  --provider openai \
  --model gpt-4o \
  --temperature 0.0 \
  --max_tokens 200

Loaded 62 QA pairs from /workspace/data/qa/llm_batch/chunks/qa_pairs_chunk_7.csv
Using provider: openai
Prepared 186 batch requests to /workspace/data/results/llm_batch/gpt-4o/chunk_7/requests.jsonl
Wrote mapping to /workspace/data/results/llm_batch/gpt-4o/chunk_7/mapping.csv


In [41]:
# Submit LLM batch job and store the output batch ID
from datetime import datetime
import subprocess

# Generate the file path for storing the batch ID
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_id_file = projectRoot / f"BATCH_ID/{timestamp}_batchID"

# Run the command and capture the batch ID

result = subprocess.run(
    [
        "python", "/workspace/src/evaluation/benchmark_runner_batch.py", "submit",
        "/workspace/data/results/llm_batch/gpt-4o/chunk_7/requests.jsonl",
        "--provider", "openai",
        "--window", "24h"
    ],
    stdout=subprocess.PIPE,
    text=True,
    check=True
)

# Extract the batch ID from the command output
batch_id = result.stdout.strip()

# Save the batch ID to the file
batch_id_file.parent.mkdir(parents=True, exist_ok=True)
with open(batch_id_file, "w") as file:
    file.write(batch_id)

print(f"Batch ID saved to: {batch_id_file}")

Batch ID saved to: /workspace/BATCH_ID/20260104_232606_batchID


In [43]:
# Check status of LLM batch job
# batch_id = "<batch_id>"  # Replace with the actual batch ID

# Read batch_id from the file
with open(batch_id_file, "r") as file:
    batch_id = file.read().strip()
batch_id = batch_id.split("id=")[1].split(",")[0]
    
!python /workspace/src/evaluation/benchmark_runner_batch.py status {batch_id} \
    --provider openai

Batch batch_695af7136b548190a03f42c1f783f99c status: completed
Counts: BatchRequestCounts(completed=186, failed=0, total=186)
Output file id: file-7uMUFAkVw49SHBmrjft2A9


In [44]:
# Collect LLM batch results
!python /workspace/src/evaluation/benchmark_runner_batch.py collect \
  {batch_id} \
  /workspace/data/results/llm_batch/gpt-4o/chunk_7/output.jsonl \
  /workspace/data/results/llm_batch/gpt-4o/chunk_7/mapping.csv \
  /workspace/data/results/llm_batch/gpt-4o/chunk_7_results.csv \
  /workspace/data/results/llm_batch/gpt-4o/chunk_7_raw.csv \
  --model gpt-4o \
  --provider openai

Saved batch output JSONL to /workspace/data/results/llm_batch/gpt-4o/chunk_7/output.jsonl
Wrote parsed results to /workspace/data/results/llm_batch/gpt-4o/chunk_7_results.csv (62 rows)
Wrote raw outputs to /workspace/data/results/llm_batch/gpt-4o/chunk_7_raw.csv (62 rows)
